# Austin Metro Economic Productivity Map — Value per Acre

Urban3 / Strong-Towns style **value-per-acre** map: every parcel extruded by its
economic value per acre, so dense central land towers over big-box / suburban tracts.

**Method (per parcel):**
- `value_per_acre   = market_value / land_acres`  — what the land is worth
- `value_per_acre_adj = value_per_acre / pvs_ratio`  — sales-ratio normalized for cross-county comparability
- `tax_per_acre     = taxable_value * effective_rate / land_acres`  — what the land pays (fiscal layer)

**Data:** per-county ArcGIS FeatureServers (value + geometry + acreage bundled, no roll-join).
Config + fetcher live in `v2_county_sources.py` / `v2_fetch_parcels.py`. Counties: Travis,
Williamson, Hays. Sources verified 2026-06-29.

**Scope toggle:** `SCOPE` below controls what gets pulled. The default is a fast downtown-Austin
slice (a bbox over Travis) for iteration; set counties to `None` to pull whole counties for the
full metro scene.

In [1]:
import sys
import math
from pathlib import Path

import geopandas as gpd
import pandas as pd
import pydeck as pdk

# Make the repo-root helper modules importable whether run from repo root or notebooks/.
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from v2_county_sources import COUNTY_SOURCES, PVS_RATIOS
from v2_fetch_parcels import fetch_county_gdf

OUT_DIR = REPO_ROOT / 'outputs'
OUT_DIR.mkdir(exist_ok=True)
print('counties available:', list(COUNTY_SOURCES))
print('pvs ratios:', PVS_RATIOS)

counties available: ['travis', 'williamson', 'hays']
pvs ratios: {'travis': 1.0, 'williamson': 0.96, 'hays': 0.97}


## Config

In [2]:
# Downtown + central Austin envelope (lon/lat, EPSG:4326). The hero/validation area.
DOWNTOWN_BBOX = (-97.78, 30.24, -97.71, 30.30)

# SCOPE: {county: bbox-or-None}. bbox = fast bounded slice; None = whole county.
# Default = downtown Travis slice. For the full metro, use:
#   SCOPE = {'travis': None, 'williamson': None, 'hays': None}
SCOPE = {'travis': DOWNTOWN_BBOX}

# Blended effective property-tax rate (city+county+school+special districts), ~2.1%.
# Approximation for the tax/acre fiscal layer; rates differ slightly per jurisdiction.
EFFECTIVE_TAX_RATE = 0.021

# 3D extrusion: tallest (p99) parcel tops out at MAX_HEIGHT meters; all else scales linearly.
MAX_HEIGHT = 2500.0

# Which metric drives color + height in the 3D scene.
ELEV_METRIC = 'value_per_acre'

## Fetch parcels

In [3]:
frames = []
for county, bbox in SCOPE.items():
    print(f'fetching {county}' + (f' (bbox {bbox})' if bbox else ' (full county)') + ' ...')
    gdf_c = fetch_county_gdf(county, bbox=bbox, only_valued=True, with_geometry=True)
    print(f'  {len(gdf_c):,} parcels')
    frames.append(gdf_c)

gdf = gpd.GeoDataFrame(pd.concat(frames, ignore_index=True), crs='EPSG:4326')
print(f'total: {len(gdf):,} parcels')
gdf.head(3)

fetching travis (bbox (-97.78, 30.24, -97.71, 30.3)) ...


  21,092 parcels
total: 21,092 parcels


,county,parcel_id,market_value,taxable_value,land_acres,land_use,geometry
0,travis,0217020203,771661,771661.0,0.1550,1 FAM DWELLING,"POLYGON ((-97.74471 30.30069, -97.74478 30.300..."
1,travis,0217020320,1137993,1137993.0,0.2002,2 FAM DWELLING,"POLYGON ((-97.74467 30.30007, -97.74446 30.299..."
2,travis,0118040508,2165099,2165099.0,0.2364,1 FAM DWELLING,"POLYGON ((-97.76193 30.30041, -97.7618 30.3003..."


## Clean + compute metrics

Drop null geometry, non-positive acreage, and zero/exempt value; coerce string acreage to float;
compute the three metrics; winsorize the color/elevation tail at p99 (distributions are heavily
right-skewed — downtown is 100x+ the suburbs).

In [4]:
g = gdf[gdf.geometry.notna()].copy()
g['land_acres'] = pd.to_numeric(g['land_acres'], errors='coerce')
g['market_value'] = pd.to_numeric(g['market_value'], errors='coerce')
g = g[(g['land_acres'] > 0) & (g['market_value'] > 0)]

g['value_per_acre'] = g['market_value'] / g['land_acres']
g['pvs_ratio'] = g['county'].map(PVS_RATIOS)
g['value_per_acre_adj'] = g['value_per_acre'] / g['pvs_ratio']

# tax/acre needs taxable_value (present for Travis; NaN where a county omits it)
if 'taxable_value' in g.columns:
    g['taxable_value'] = pd.to_numeric(g['taxable_value'], errors='coerce')
    g['tax_per_acre'] = g['taxable_value'] * EFFECTIVE_TAX_RATE / g['land_acres']
else:
    g['tax_per_acre'] = float('nan')

# keep finite, sane values
g = g[g['value_per_acre'].between(1, 1e12)]
print(f'{len(g):,} parcels after cleaning')
g[['value_per_acre', 'value_per_acre_adj', 'tax_per_acre']].describe()

21,092 parcels after cleaning


,value_per_acre,value_per_acre_adj,tax_per_acre
count,2.109200e+04,2.109200e+04,2.109100e+04
mean,7.663172e+06,7.663172e+06,1.609278e+05
std,1.452750e+07,1.452750e+07,3.050847e+05
min,1.296092e+01,1.296092e+01,2.721794e-01
25%,3.876511e+06,3.876511e+06,8.140427e+04
50%,5.440430e+06,5.440430e+06,1.142391e+05
75%,8.067779e+06,8.067779e+06,1.694318e+05
max,4.836484e+08,4.836484e+08,1.015662e+07


## Validation — the Urban3 sanity story
Top parcels by value/acre should be dense central commercial / downtown; bottom should be
large low-value tracts, parking, and fringe land.

In [5]:
cols = ['county', 'parcel_id', 'land_use', 'market_value', 'land_acres', 'value_per_acre']
cols = [c for c in cols if c in g.columns]
print('TOP 10 by value/acre')
display(g.sort_values('value_per_acre', ascending=False)[cols].head(10))
print('BOTTOM 10 by value/acre')
display(g.sort_values('value_per_acre')[cols].head(10))

TOP 10 by value/acre


,county,parcel_id,land_use,market_value,land_acres,value_per_acre
4970,travis,0206011606,OFF HI-RISE >= 6,196119412,0.4055,4.836484e+08
19202,travis,0206011205,LUXURY HI-RISE APTS 100+,192780000,0.4055,4.754131e+08
5009,travis,0205021001,HIRISE CONDO/APT,188356279,0.4055,4.645038e+08
20556,travis,0203031001,LUXURY HI-RISE APTS 100+,233950000,0.5405,4.328400e+08
20013,travis,0206011901,OFF HI-RISE >= 6,387556329,0.9377,4.133052e+08
9528,travis,0402010201,NaN,431250,0.0011,3.920455e+08
20249,travis,0206030709,HOTEL-FULL SERVC,98000000,0.2535,3.865878e+08
14965,travis,0206030816,HOTEL-FULL SERVC,58000000,0.1593,3.640929e+08
3277,travis,0214011303,APARTMENT 100+,143490000,0.4014,3.574738e+08
1512,travis,0210021714,OFF HI-RISE >= 6,147991225,0.4406,3.358857e+08


BOTTOM 10 by value/acre


,county,parcel_id,land_use,market_value,land_acres,value_per_acre
17112,travis,0104090221,NaN,20,1.5431,12.960923
18099,travis,0217130103,NaN,20000,30.1221,663.964332
18103,travis,0219120242,NaN,10000,13.1408,760.988676
14167,travis,0215080169,NaN,210,0.0483,4347.826087
10539,travis,NaN,RETAIL STORE,750000,163.6395,4583.245488
20589,travis,0105000103,NaN,750000,163.6395,4583.245488
16410,travis,0214000102,NaN,1500,0.3000,5000.000000
20971,travis,0405061601,NaN,26250,3.9209,6694.891479
10934,travis,0404070326,NaN,5478,0.6130,8936.378467
16176,travis,0404070615,Detail Only,7699,0.7708,9988.323819


## 3D extruded scene (the Urban3 look)
pydeck `PolygonLayer`, extruded + colored by the chosen metric on a log scale
(blue = low → yellow → red = high), dark basemap. Writes a standalone HTML to `outputs/`.

In [6]:
# Winsorize the metric for color + elevation
metric = g[ELEV_METRIC]
p99 = metric.quantile(0.99)
p01 = max(metric.quantile(0.01), 1.0)
lo, hi = math.log(p01), math.log(p99)
ELEV_SCALE = MAX_HEIGHT / p99
print(f'{ELEV_METRIC}: p01={p01:,.0f}  p99={p99:,.0f}  (tallest -> {MAX_HEIGHT:.0f} m)')

def lognorm(v):
    x = (math.log(max(v, 1)) - lo) / (hi - lo)
    return min(max(x, 0.0), 1.0)

# Color ramp: blue (low) -> yellow -> red (high)
STOPS = [(0.0, (30, 60, 160)), (0.5, (240, 220, 60)), (1.0, (200, 30, 30))]
def ramp(t):
    for i in range(len(STOPS) - 1):
        t0, c0 = STOPS[i]; t1, c1 = STOPS[i + 1]
        if t <= t1:
            f = (t - t0) / (t1 - t0) if t1 > t0 else 0
            return [int(c0[j] + f * (c1[j] - c0[j])) for j in range(3)]
    return list(STOPS[-1][1])

def rings(geom):
    if geom.geom_type == 'Polygon':
        return [list(geom.exterior.coords)]
    if geom.geom_type == 'MultiPolygon':
        return [list(p.exterior.coords) for p in geom.geoms]
    return []

poly_data = []
for _, row in g.iterrows():
    v = row[ELEV_METRIC]
    color = ramp(lognorm(v))
    v_clip = min(v, p99)
    for ring in rings(row.geometry):
        poly_data.append({
            'polygon': [[x, y] for x, y in ring],
            'elevation': v_clip * ELEV_SCALE,
            'color': color,
            'vpa': f"${row['value_per_acre']:,.0f}/acre",
            'mkt': f"${row['market_value']:,.0f}",
            'acres': f"{row['land_acres']:.3f}",
        })
print(f'{len(poly_data):,} polygon rings')

value_per_acre: p01=953,874  p99=42,992,113  (tallest -> 2500 m)


21,232 polygon rings


In [7]:
# Center the camera on the data's centroid
cx = float(g.geometry.union_all().centroid.x)
cy = float(g.geometry.union_all().centroid.y)

layer = pdk.Layer(
    'PolygonLayer', poly_data,
    get_polygon='polygon', get_elevation='elevation', get_fill_color='color',
    extruded=True, wireframe=False, pickable=True, elevation_scale=1,
)
view = pdk.ViewState(latitude=cy, longitude=cx, zoom=13.0, pitch=55, bearing=20)
deck = pdk.Deck(
    layers=[layer], initial_view_state=view, map_style='dark',
    tooltip={'text': '{vpa}\nMarket: {mkt}\nAcres: {acres}'},
)
out_html = OUT_DIR / 'map_value_per_acre_metro_3d.html'
deck.to_html(str(out_html), notebook_display=False)
print('wrote', out_html)

wrote /Users/chaseeasterling/GitHub/fire-incident-analysis/outputs/map_value_per_acre_metro_3d.html


In [8]:
from IPython.display import IFrame
# Lightweight: references the file on disk (does not embed the data in the notebook).
IFrame(src=str(out_html.relative_to(REPO_ROOT)) if REPO_ROOT in out_html.parents else str(out_html),
       width='100%', height=600)

## Next: scale to the full metro
Set `SCOPE = {'travis': None, 'williamson': None, 'hays': None}` and re-run. Two things this
downtown slice already proved are needed at metro scale (~638k parcels):
- **Geometry simplification** before building `poly_data` (the slice's 21k parcels = 25 MB HTML;
  638k unsimplified would be ~750 MB and won't open). Simplify in a projected CRS, e.g.
  `g['geometry'] = g.to_crs(2277).geometry.simplify(5).to_crs(4326)`.
- **H3 hex aggregation** for a separate lightweight 2D shareable map (folium can't draw ~1M polygons).